# SAE Scoping Pipeline

This notebook orchestrates the full SAE scoping pipeline by calling the scripts in the `scripts/` directory.
No wandb account is needed — logging is disabled by default.

## Steps
1. **Find firing rates** — run `scripts/find_firing_rates.py` to collect per-feature firing statistics from the SAE across your chosen datasets.
2. **Plot firing rates** — run `scripts/plot_firing_rates.py` to visualise the distributions and choose a pruning threshold.
3. **Train with firing rates** — run `scripts/train_with_firing_rates.py` to fine-tune Gemma-2 9B with the pruned SAE hooked in.

Each step delegates entirely to the corresponding script, so this notebook is just a thin orchestration layer.

## Setup

Install the package (if not already installed) so the `sae_scoping` library is importable by the scripts.

In [ ]:
# Install the package in editable mode so scripts can import sae_scoping
!pip install -e . -q

## Step 1 — Find Firing Rates

Runs the model over the selected datasets and saves per-feature firing-rate distributions to `scripts/.cache/`.

**Options** (passed directly to `find_firing_rates.py`):
- `-d` — comma-separated dataset names (choices: `chemistry`, `apps`, `ultrachat`)
- `-i` — comma-separated `ignore_padding` booleans (e.g. `True,False`)
- `-b` — batch size

Adjust the variables below and run the cell.

In [ ]:
DATASETS = "chemistry"       # e.g. "chemistry,apps,ultrachat"
IGNORE_PADDINGS = "True"     # e.g. "True,False"
BATCH_SIZE = 7

!python scripts/find_firing_rates.py \
    -d {DATASETS} \
    -i {IGNORE_PADDINGS} \
    -b {BATCH_SIZE}

## Step 2 — Plot Firing Rates

Reads the cached distributions produced in Step 1 and writes PNG plots to `scripts/.cache/plots/`.

**Options** (passed directly to `plot_firing_rates.py`):
- `-c` — path to the cache directory (default: `scripts/.cache`)
- `-i` — which `ignore_padding` split to load (`True` or `False`)
- `-k` — top-K features used for the overlap plot

In [ ]:
CACHE_DIR = "scripts/.cache"
IGNORE_PADDING = "True"
TOP_K = 500

!python scripts/plot_firing_rates.py \
    -c {CACHE_DIR} \
    -i {IGNORE_PADDING} \
    -k {TOP_K}

### Inspect the plots

In [ ]:
from pathlib import Path
from IPython.display import Image, display

plot_dir = Path(CACHE_DIR) / "plots"
for png in sorted(plot_dir.glob("*.png")):
    print(png.name)
    display(Image(filename=str(png)))

## Step 3 — Train with Firing Rates

Fine-tunes Gemma-2 9B with the pruned SAE injected at the chosen hook-point.  
Set `DIST_PATH` to the `distribution.safetensors` file produced in Step 1 (or `"vanilla"` to skip the SAE entirely).

**Key options** (passed directly to `train_with_firing_rates.py`):
- `-p` — path to `distribution.safetensors` (required, or `"vanilla"`)
- `-t` — dataset name to train on (must match one defined inside the script)
- `-b` — per-device batch size
- `-a` — gradient accumulation steps
- `-s` — maximum training steps
- `-h` — firing-rate threshold for neuron pruning (required when `-p vanilla` or when no checkpoint is given)
- `-c` — optional checkpoint to resume from
- `-hook` — override the hook-point layer (useful for vanilla runs)
- `-se` — save every N steps
- `-sl` — save total limit
- `-ml` — maximum sequence length
- `-e` — comma-delimited list of eval dataset names (default: all)
- `-ts` — number of evaluation samples per dataset

In [ ]:
# Path to the distribution.safetensors saved by Step 1, or "vanilla" to skip SAE.
# Example: "scripts/.cache/ignore_padding_True/chemistry/layer_20--width_16k--canonical/distribution.safetensors"
DIST_PATH = "scripts/.cache/ignore_padding_True/chemistry/layer_20--width_16k--canonical/distribution.safetensors"

TRAIN_DATASET = "chemistry"  # must be a dataset name defined in the script
BATCH_SIZE = 2
ACCUM = 8
MAX_STEPS = 1000
THRESHOLD = 1e-4   # firing-rate threshold; only needed when not inferring from a checkpoint

!python scripts/train_with_firing_rates.py \
    -p "{DIST_PATH}" \
    -t {TRAIN_DATASET} \
    -b {BATCH_SIZE} \
    -a {ACCUM} \
    -s {MAX_STEPS} \
    -h {THRESHOLD}